In [2]:
#Carga del dataset
import pandas as pd
import numpy as np
from pathlib import Path

project_root = Path.cwd().parent.parent
data_path = project_root / 'data' / 'raw' / 'books.csv'

df = pd.read_csv(data_path)


print(f"Dataset original: {df.shape[0]} libros, {df.shape[1]} columnas") #Obtener filas y columnas a raiz de tuplas, siendo 0 las filas y 1 las columnas

Dataset original: 6810 libros, 12 columnas


In [6]:
print("Columnas del dataset:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:2d}. {col}")

Columnas del dataset:
 1. isbn13
 2. isbn10
 3. title
 4. subtitle
 5. authors
 6. categories
 7. thumbnail
 8. description
 9. published_year
10. average_rating
11. num_pages
12. ratings_count


In [7]:
#Identificacion de problemas
print("PROBLEMAS A RESOLVER\n")
print(f"1. Libros sin categoría (target): {df['categories'].isnull().sum()}")
print(f"2. Valores nulos en otras columnas:")
print(df.isnull().sum()[df.isnull().sum() > 0])

PROBLEMAS A RESOLVER

1. Libros sin categoría (target): 99
2. Valores nulos en otras columnas:
subtitle          4429
authors             72
categories          99
thumbnail          329
description        262
published_year       6
average_rating      43
num_pages           43
ratings_count       43
dtype: int64


In [11]:
#DECISION: eliminar libros sin categoría
#Razón: No podemos predecir la categoría sin saber cuál es

print(f"Antes: {len(df)} libros")

df_clean = df[df['categories'].notna()].copy()

print(f"Después: {len(df_clean)} libros")
print(f"Eliminados: {len(df) - len(df_clean)} libros")

Antes: 6810 libros
Después: 6711 libros
Eliminados: 99 libros


In [12]:
# Verificar que no quedan nulos en categories
print(f"Nulos en categories: {df_clean['categories'].isnull().sum()}")

Nulos en categories: 0


In [15]:
# Ver distribución de categorías/géneros
genre_counts = df_clean['categories'].value_counts()

print(f"Géneros únicos: {len(genre_counts)}")
print(f"\nTop 15 géneros: ")
print(genre_counts.head(15))

print(f"\n¿Cuántos géneros tienen menos de 10 libros?")
raros = (genre_counts < 10).sum()
print(f"{raros} géneros con menos de 10 libros")

Géneros únicos: 567

Top 15 géneros: 
categories
Fiction                      2588
Juvenile Fiction              538
Biography & Autobiography     401
History                       264
Literary Criticism            166
Philosophy                    160
Comics & Graphic Novels       159
Religion                      137
Drama                         132
Juvenile Nonfiction           116
Poetry                         79
Science                        71
Literary Collections           71
Business & Economics           67
Social Science                 60
Name: count, dtype: int64

¿Cuántos géneros tienen menos de 10 libros?
518 géneros con menos de 10 libros


In [18]:
#Eliminar libros sin autor

# Ver cuántos nulos hay para autores
print(f"Libros sin autor: {df_clean['authors'].isnull().sum()}")

# Limpiar los nulos
df_clean = df_clean[df_clean['authors'].notna()].copy()

print(f"Libros después de eliminar sin autor: {len(df_clean)}")


Libros sin autor: 71
Libros después de eliminar sin autor: 6640


In [23]:
#Eliminar nulos de columnas numéricas average, num_pages y ratings_count

print("Nulos en features numéricas:")
print(f" average_rating: {df_clean['average_rating'].isnull().sum()}")
print(f" num_pages: {df_clean['num_pages'].isnull().sum()}")
print(f" ratings_count: {df_clean['ratings_count'].isnull().sum()}")

# Limpiar los nulos
df_clean = df_clean.dropna(subset=['average_rating', 'num_pages', 'ratings_count'])

print(f"\nLibros después de limpiar nulos: {len(df_clean)}")


Nulos en features numéricas:
 average_rating: 40
 num_pages: 40
 ratings_count: 40

Libros después de limpiar nulos: 6600


In [26]:
#Eliminar nulos en published_year

print("Nulos en published_year:")
print(f" published_year: {df_clean['published_year'].isnull().sum()}")

# Limpiar nulos
df_clean = df_clean[df_clean['published_year'].notna()].copy()

print(f"Libros después de limpiar año: {len(df_clean)}")

Nulos en published_year:
 published_year: 1
Libros después de limpiar año: 6599


In [27]:
#Limpiar años raros

# Ver rango de años
print("Estadísticas de años:")
print(df_clean['published_year'].describe())

# Filtrar años razonables entre 1900 y 2026
df_clean = df_clean [
    (df_clean['published_year'] >= 1900) &
    (df_clean['published_year'] <= 2026)
].copy()

print(f"\nLibros con año válido: {len(df_clean)}")

Estadísticas de años:
count    6599.000000
mean     1998.750417
std        10.168465
min      1876.000000
25%      1997.000000
50%      2002.000000
75%      2005.000000
max      2019.000000
Name: published_year, dtype: float64

Libros con año válido: 6597


In [32]:
#Crear nueva feature: longitud del título
df_clean['title_length'] = df_clean['title'].str.len()

print("Estadísticas de longitud de títulos:")
print(df_clean['title_length'].describe())

Estadísticas de longitud de títulos:
count    6597.000000
mean       20.680309
std        11.801476
min         1.000000
25%        13.000000
50%        18.000000
75%        25.000000
max       145.000000
Name: title_length, dtype: float64


In [33]:
print("=" * 60)
print("RESUMEN DESPUÉS DE LIMPIEZA")
print("=" * 60)

print(f"\nLibros originales: 6,810")
print(f"Libros finales: {len(df_clean)}")
print(f"Eliminados: {6810 - len(df_clean)} libros")

print(f"\nGéneros únicos: {df_clean['categories'].nunique()}")

print(f"\nNueva columna creada:")
print("  - title_length")

print(f"\nValores nulos restantes:")
nulos = df_clean.isnull().sum()
if nulos.sum() == 0:
    print("  ✅ Ninguno")
else:
    print(nulos[nulos > 0])

RESUMEN DESPUÉS DE LIMPIEZA

Libros originales: 6,810
Libros finales: 6597
Eliminados: 213 libros

Géneros únicos: 563

Nueva columna creada:
  - title_length

Valores nulos restantes:
subtitle       4281
thumbnail       259
description     188
dtype: int64


In [34]:
# Guardar dataset limpio
output_path = '../../data/processed/books_clean_phase1.csv'

df_clean.to_csv(output_path, index=False)

print(f"✅ Dataset limpio guardado en:")
print(f"   {output_path}")
print(f"\n📊 Resumen del archivo guardado:")
print(f"   Libros: {len(df_clean)}")
print(f"   Columnas: {df_clean.shape[1]}")

✅ Dataset limpio guardado en:
   ../../data/processed/books_clean_phase1.csv

📊 Resumen del archivo guardado:
   Libros: 6597
   Columnas: 15
